# Chapter 13 — The External Sector

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch13_external_sector.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 13.1 Three exchange rates

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from indiadata import bis, worldbank, mospi_nas

usd = bis("WS_XRU", "M.IN.INR.A").set_index("date")["value"].rename("rs_per_usd")
neer = bis("WS_EER", "M.N.B.IN").set_index("date")["value"].rename("neer")
reer = bis("WS_EER", "M.R.B.IN").set_index("date")["value"].rename("reer")

for s in (usd, neer, reer):
    print(f"{s.name:11} {s.index.min():%b %Y} to {s.index.max():%b %Y}, "
          f"latest {s.iloc[-1]:.2f}")
print(f"Average Rs per US$ in 1991: {usd['1991'].mean():.2f}")

In [ ]:
BLUE, ORANGE, LIGHT = "#1f4e79", "#c55a11", "#9db7d5"

fig, ax = plt.subplots(figsize=(10, 4.6))
ax.plot(usd.index, usd, color=BLUE, linewidth=1.6)
for date, text, dy in [("1966-06-01", "1966\ndevaluation", 12),
                       ("1991-07-01", "1991\ncrisis", 14),
                       ("2013-08-01", "2013 taper\ntantrum", -26),
                       ("2022-10-01", "2022", -22)]:
    t = pd.Timestamp(date)
    ax.annotate(text, xy=(t, usd[t]), xytext=(t, usd[t] + dy), fontsize=9,
                ha="center", arrowprops=dict(arrowstyle="->", color="grey"))
ax.set_title("Rupees per US dollar, 1957–2026", loc="left", fontsize=13)
ax.set_ylabel("Rs per US$ (monthly average)")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.02, "Source: BIS, bilateral exchange rates (WS_XRU). "
         "A rise is a depreciation of the rupee.", fontsize=8, color="grey")
plt.show()

## 13.2 Depreciation and volatility

In [ ]:
def avg_depreciation(series, start, end):
    """Average annual % change between two months (compound rate)."""
    a, b = series[start], series[end]
    years = (pd.Timestamp(end) - pd.Timestamp(start)).days / 365.25
    return ((b / a) ** (1 / years) - 1) * 100


for start in ["1995-01-01", "2005-01-01", "2015-01-01"]:
    print(f"{start[:4]}-2026: Rs/$ from {usd[start]:6.2f} to {usd.iloc[-1]:6.2f}, "
          f"average depreciation {avg_depreciation(usd, start, usd.index[-1]):.1f}% a year")

In [ ]:
monthly = usd.pct_change() * 100                    # % change on last month
print("Standard deviation of monthly changes (%):")
for a, b in [("1995", "2004"), ("2005", "2014"), ("2015", "2026")]:
    print(f"  {a}-{b}: {monthly[a:b].std():.2f}")

vol = monthly.rolling(12).std()
print(f"\nMost volatile 12 months since 1995 end in {vol['1995':].idxmax():%B %Y}, "
      f"std = {vol['1995':].max():.2f}%")

In [ ]:
three_month = (usd / usd.shift(3) - 1) * 100
three_month["1993":].nlargest(5).round(1)

## 13.3 Event windows: 2013 and 2022

In [ ]:
def window(series, start, months=13):
    """Index the series to 100 in month `start`, for `months` months."""
    seg = series[start:].iloc[:months]
    return pd.Series((seg / seg.iloc[0] * 100).to_numpy(), index=range(months))


episodes = pd.DataFrame({
    "2013: Rs/$": window(usd, "2013-04-01"),
    "2013: NEER": window(neer, "2013-04-01"),
    "2022: Rs/$": window(usd, "2021-12-01"),
    "2022: NEER": window(neer, "2021-12-01"),
})
episodes.loc[[0, 4, 8, 12]].round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, year, start in [(axes[0], "2013", "Apr 2013"), (axes[1], "2022", "Dec 2021")]:
    ax.plot(episodes[f"{year}: Rs/$"], color=BLUE, marker="o",
            label="Rs per US$ (up = rupee weaker)")
    ax.plot(episodes[f"{year}: NEER"], color=ORANGE, marker="o",
            label="NEER (down = rupee weaker)")
    ax.axhline(100, color="black", linewidth=0.8)
    ax.set_title(f"{year} (index, {start} = 100)", loc="left", fontsize=12)
    ax.set_xlabel(f"Months after {start}")
    ax.spines[["top", "right"]].set_visible(False)
axes[1].legend(frameon=False, fontsize=8, loc="lower left")
fig.text(0.01, -0.04, "Source: BIS, bilateral and broad effective exchange "
         "rates.", fontsize=8, color="grey")
plt.tight_layout()
plt.show()

## 13.4 Nominal and real: the effective rates

In [ ]:
eer = pd.concat([neer, reer], axis=1).dropna()
eer_94 = eer / eer.iloc[0] * 100
print(f"{eer.index[-1]:%B %Y}, index Jan 1994 = 100:")
print(eer_94.iloc[-1].round(1))
peak = eer_94.loc["2023":, "reer"].idxmax()
print(f"\nREER peak since 2023: {eer_94.loc[peak, 'reer']:.1f} in {peak:%B %Y}; "
      f"change to {eer.index[-1]:%b %Y}: "
      f"{(eer.iloc[-1]['reer'] / eer.loc[peak, 'reer'] - 1) * 100:.1f}%")
above = (eer_94.loc["2007":"2024", "reer"] > 100).mean()
print(f"Share of months in 2007-2024 with the REER above its Jan 1994 level: {above:.0%}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.3))
ax.plot(eer_94.index, eer_94["neer"], color=ORANGE, linewidth=1.6,
        label="Nominal effective (NEER)")
ax.plot(eer_94.index, eer_94["reer"], color=BLUE, linewidth=1.6,
        label="Real effective (REER)")
ax.axhline(100, color="black", linewidth=0.8)
ax.set_title("The rupee against a basket of 64 economies' currencies",
             loc="left", fontsize=13)
ax.set_ylabel("Index, Jan 1994 = 100 (up = stronger rupee)")
ax.legend(frameon=False, fontsize=9, loc="lower left")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.02, "Source: BIS, broad effective exchange rate indices "
         "(WS_EER, rebased).", fontsize=8, color="grey")
plt.show()

## 13.5 An approximate real rupee–dollar rate

In [ ]:
cpi = (worldbank("FP.CPI.TOTL", "IND;USA")
       .pivot(index="year", columns="iso3", values="value"))
annual = usd.resample("YE").mean()
annual.index = annual.index.year

real = pd.DataFrame({"S": annual, "P_IN": cpi["IND"], "P_US": cpi["USA"]}).dropna()
real = real.loc[1993:]
real["q"] = real["S"] * real["P_US"] / real["P_IN"]
real["q_index"] = real["q"] / real.loc[1993, "q"] * 100
real["S_index"] = real["S"] / real.loc[1993, "S"] * 100
print("Years:", real.index.min(), "to", real.index.max())
print(f"Index 1993 = 100 in {real.index[-1]}: nominal {real['S_index'].iloc[-1]:.0f}, "
      f"real {real['q_index'].iloc[-1]:.0f}")
real[["S", "P_IN", "P_US", "q"]].iloc[[0, 10, 20, -1]].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.3))
ax.plot(real.index, real["S_index"], color=ORANGE, marker="o", markersize=4,
        label="Nominal Rs per US$")
ax.plot(real.index, real["q_index"], color=BLUE, marker="o", markersize=4,
        label="Real Rs per US$ (CPI-adjusted)")
ax.axhline(100, color="black", linewidth=0.8)
ax.set_title("Nominal and real rupee–dollar rates (1993 = 100)", loc="left",
             fontsize=13)
ax.set_ylabel("Index (up = weaker rupee)")
ax.legend(frameon=False, fontsize=9, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.02, "Sources: BIS (annual average of monthly Rs/US$); World "
         "Bank WDI, consumer price indices (FP.CPI.TOTL).", fontsize=8,
         color="grey")
plt.show()

In [ ]:
import statsmodels.formula.api as smf

ppp = pd.DataFrame({
    "depreciation": real["S"].pct_change() * 100,
    "inflation_gap": (real["P_IN"].pct_change() - real["P_US"].pct_change()) * 100,
}).dropna()
fit = smf.ols("depreciation ~ inflation_gap", data=ppp).fit(cov_type="HC1")
print(fit.summary().tables[1])
print(f"Years: {int(fit.nobs)},  R-squared: {fit.rsquared:.2f}")
print(f"Average depreciation {ppp['depreciation'].mean():.2f}%, "
      f"average inflation gap {ppp['inflation_gap'].mean():.2f} points")

## 13.6 How open is India? Trade in the national accounts

In [ ]:
nas = {k: mospi_nas(k).set_index("year")["current_price"]
       for k in ["exports", "imports", "GDP"]}
trade = pd.DataFrame(nas)
trade["exports_gdp"] = trade["exports"] / trade["GDP"] * 100
trade["imports_gdp"] = trade["imports"] / trade["GDP"] * 100
trade["openness"] = trade["exports_gdp"] + trade["imports_gdp"]
trade["net_exports_gdp"] = trade["exports_gdp"] - trade["imports_gdp"]
trade[["exports_gdp", "imports_gdp", "openness", "net_exports_gdp"]].round(1)

## 13.7 The current account and how it is paid for

In [ ]:
series = {"cab_gdp": "BN.CAB.XOKA.GD.ZS",        # current account, % GDP
          "fdi_gdp": "BX.KLT.DINV.WD.GD.ZS",     # FDI net inflows, % GDP
          "remit_gdp": "BX.TRF.PWKR.DT.GD.ZS",   # remittances received, % GDP
          "remit_usd": "BX.TRF.PWKR.CD.DT",      # remittances received, US$
          "reserves_usd": "FI.RES.TOTL.CD"}      # total reserves incl. gold, US$
bop = pd.DataFrame({name: worldbank(code).set_index("year")["value"]
                    for name, code in series.items()})
bop["remit_bn"] = bop["remit_usd"] / 1e9
bop["reserves_bn"] = bop["reserves_usd"] / 1e9
bop = bop.drop(columns=["remit_usd", "reserves_usd"])
bop.loc[2005:].round(2)

In [ ]:
print("Largest current-account deficit since 1990:",
      f"{bop.loc[1990:, 'cab_gdp'].min():.2f}% of GDP in {bop.loc[1990:, 'cab_gdp'].idxmin()}")
print("Years with a current-account surplus since 1990:",
      list(bop.loc[1990:].index[bop.loc[1990:, "cab_gdp"] > 0]))
print(f"Reserves: {bop.loc[2012, 'reserves_bn']:.0f} bn US$ in 2012, "
      f"{bop.loc[2021, 'reserves_bn']:.0f} in 2021, {bop.loc[2022, 'reserves_bn']:.0f} "
      f"in 2022, {bop['reserves_bn'].iloc[-1]:.0f} in {bop.index[-1]}")

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4.3))
b = bop.loc[1995:]
ax1.bar(b.index, b["cab_gdp"], color=[ORANGE if v < 0 else BLUE for v in b["cab_gdp"]],
        label="Current account balance, % of GDP (left)")
ax1.axhline(0, color="black", linewidth=0.8)
ax1.set_ylabel("% of GDP")
ax2 = ax1.twinx()
ax2.plot(b.index, b["reserves_bn"], color="grey", linewidth=2,
         label="Foreign-exchange reserves, US$ bn (right)")
ax2.set_ylabel("US$ billion")
ax1.set_title("India's current account and reserves", loc="left", fontsize=13)
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, frameon=False, fontsize=9, loc="upper left")
ax1.set_ylim(-6, 6)
for ax in (ax1, ax2):
    ax.spines[["top"]].set_visible(False)
fig.text(0.01, -0.02, "Source: World Bank WDI (BN.CAB.XOKA.GD.ZS, "
         "FI.RES.TOTL.CD; reserves include gold).", fontsize=8, color="grey")
plt.show()

## 13.8 Remittances: India's steadiest inflow

In [ ]:
codes = "IND;MEX;CHN;PHL;FRA;EGY;PAK;BGD;DEU;NGA;WLD"
rem = worldbank("BX.TRF.PWKR.CD.DT", codes)
year = rem[rem["iso3"] != "WLD"].groupby("iso3")["year"].max().min()
latest = rem[rem["year"] == year].set_index("country")["value"] / 1e9
world = latest.pop("World")
latest = latest.sort_values()
print(f"Personal remittances received, {year}, US$ bn")
print(latest.round(1).sort_values(ascending=False).to_string())
print(f"\nIndia's share of the world total: {latest['India'] / world:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.3))
ax.barh(latest.index, latest, color=[ORANGE if c == "India" else LIGHT
                                     for c in latest.index])
for i, v in enumerate(latest):
    ax.text(v + 1.5, i, f"{v:.0f}", va="center", fontsize=9)
ax.set_xlabel("US$ billion")
ax.set_title(f"Personal remittances received, {year}", loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Source: World Bank WDI (BX.TRF.PWKR.CD.DT). Ten large "
         "recipients chosen for comparison.", fontsize=8, color="grey")
plt.show()

## 13.9 Answering the question

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading